# `align` — one book: Sino-Nom `Label.txt` + Vietnamese `Label.txt` to `<book>.csv`, `.xlsx` and a report

Environment: a **CPU** runtime (no GPU). Python side is the package itself (`click`, `openpyxl`, both already on Colab). Lexicon tables (`entries/*.jsonl`, `lab_dict.csv` or `lexicon.sqlite`) are read from `/content`; the only network use is the package's online lookup of unknown characters.

**Status: UNVERIFIED on hardware.** Rewritten to the `ocr-engines` notebook shape; the CLI itself is tested, but this notebook has NOT been run on Colab yet: the first clean run is the check.

**Flow:** CONFIG (edit, re-run after a restart) → SETUP (clone + install) → RUN → REVIEW → ENV.
Unlike the engine notebooks, input and output are plain files on `/content` (the package never touches a bucket, a journal or `ocr-flow`: the files are the contract). `ocr-flow` is used here only to clone and install this repo (`bootstrap.setup`). Use a **fresh CPU runtime** (Runtime > Change runtime type > None).

Run it once per book. Edit only the CONFIG cell; copy `OUT_DIR` where you need it when done.

In [ ]:
# CONFIG — the only cell you edit. Re-run it after every runtime restart (variables are lost).

# ── Secrets: Colab Secrets (key icon, "Notebook access" ON). Never type or print a token. ──
from google.colab import userdata


def _secret(name):
    try:
        return userdata.get(name)
    except Exception as e:
        raise RuntimeError(f"Colab secret {name!r} unavailable ({type(e).__name__}): add it under the key icon and turn Notebook access ON") from None
GITHUB_TOKEN = _secret("GITHUB_TOKEN")   # clones the private code repos

# ── Code ──
GIT_REPO  = "obj-see/ocr-nom-align"
GIT_BRANCH = "main"                      # push your changes first: the clone is fresh each runtime
REPO_DIR  = "/content/ocr-nom-align"
FLOW_REPO = "obj-see/ocr-flow"          # shared core, used only for bootstrap/envinfo; cloned first
FLOW_BRANCH = "main"
FLOW_DIR  = "/content/ocr-flow"

# ── Book: two Label.txt files (+ optional image map) ──
BOOK = "Book"                           # files are <OUT_DIR>/<BOOK>.csv, .xlsx, _align_report.json
CH_LABEL = "/content/ch/Label.txt"      # Sino-Nom side
VI_LABEL = "/content/vi/Label.txt"      # Vietnamese side
IMAGE_MAP = ""                          # "/content/Book_image_map.json" or "" (pages paired by name)
TABLES = ["/content/lexicon/entries"]   # entries/*.jsonl (file or folder), lab_dict.csv, lexicon.sqlite
NAMES = ""                              # optional names.txt
MODE = "phonetic"                       # the book type, declared: phonetic | meaning (mixed book: --ranges)

# ── Run control ──
DETAIL = True                           # True = also write the per-sentence detail (--detail)
SCORE_MIN = 95                          # REVIEW: `score` exits 1 under this similarity (%)

# ── Output ──
OUT_DIR  = "/content/out"               # <BOOK>.csv, <BOOK>.xlsx, <BOOK>_align_report.json

## SETUP

In [ ]:
# BOOTSTRAP — fresh clones of the two code repos and their installs, in one call. Run on every new runtime.
import os
import shutil
import sys

try:
    _ = (GITHUB_TOKEN, REPO_DIR, FLOW_DIR)
except NameError:
    raise RuntimeError("Run the CONFIG cell first: variables are lost on restart.") from None
# The only repeated code a notebook may hold: ocr-flow cannot import its own bootstrap before it is cloned.
if os.path.isdir(FLOW_DIR):
    shutil.rmtree(FLOW_DIR)
import io
import tarfile

import requests  # a redirect to another host drops the Authorization header

_H = {"Authorization": f"Bearer {GITHUB_TOKEN}"}  # token only in a header: not in a URL, argv or .git/config
_API = f"https://api.github.com/repos/{FLOW_REPO}"
_c = requests.get(f"{_API}/commits/{FLOW_BRANCH}", headers=_H, timeout=60)
_c.raise_for_status()
_sha = _c.json()["sha"]
_t = requests.get(f"{_API}/tarball/{_sha}", headers=_H, timeout=120)
_t.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(_t.content)) as _tf:  # the archive has one top folder: drop it
    for _m in _tf.getmembers():
        _p = _m.name.split("/", 1)
        if len(_p) == 2 and _p[1] and ".." not in _p[1].split("/") and not _m.islnk() and not _m.issym():
            _m.name = _p[1]
            _tf.extract(_m, FLOW_DIR)
with open(f"{FLOW_DIR}/.commit-sha", "w") as _f:  # no .git: ocr_flow.colab.head_sha reads this
    _f.write(_sha + "\n")
sys.path.insert(0, f"{FLOW_DIR}/src")
from ocr_flow import bootstrap

bootstrap.setup(
    [bootstrap.RepoSpec(FLOW_REPO, FLOW_BRANCH, FLOW_DIR), bootstrap.RepoSpec(GIT_REPO, GIT_BRANCH, REPO_DIR)],
    token=GITHUB_TOKEN, gpu=False,
)   # clones ocr-flow then this repo, puts both on sys.path, installs both, checks the flow version (no GPU needed here)

In [ ]:
# SETUP check — the package imports, the CLI is on PATH, the input files and the reading tables exist.
import os
import shutil

import ocr_nom_align

assert shutil.which("ocr-nom-align"), "ocr-nom-align is not on PATH: re-run BOOTSTRAP"
print("ocr-nom-align", ocr_nom_align.__version__)
need = {"CH_LABEL": CH_LABEL, "VI_LABEL": VI_LABEL}
need |= {f"TABLES[{i}]": t for i, t in enumerate(TABLES)}
need |= {k: v for k, v in {"IMAGE_MAP": IMAGE_MAP, "NAMES": NAMES}.items() if v}
missing = {k: v for k, v in need.items() if not os.path.exists(v)}
assert not missing, f"upload these first (Colab Files panel) or fix CONFIG: {missing}"
assert MODE in ("phonetic", "meaning"), f"MODE {MODE!r}"

## RUN · REVIEW · ENV

In [ ]:
# RUN — align the book: writes <OUT_DIR>/<BOOK>.csv, .xlsx and <BOOK>_align_report.json. Stops with the CLI's message on failure.
import shlex
import subprocess

args = ["ocr-nom-align", "run", "--ch", CH_LABEL, "--vi", VI_LABEL, "--book", BOOK, "--out-dir", OUT_DIR, "--mode", MODE]
if DETAIL:
    args.append("--detail")
for t in TABLES:
    args += ["--table", t]
if IMAGE_MAP:
    args += ["--image-map", IMAGE_MAP]
if NAMES:
    args += ["--names", NAMES]
print(shlex.join(args))
r = subprocess.run(args, capture_output=True, text=True)
print(r.stdout, r.stderr)
if r.returncode:
    raise SystemExit(f"ocr-nom-align run failed (exit {r.returncode}): see the message above")

In [ ]:
# REVIEW — validate the CSV and the XLSX, then the similarity score. Re-run after any RUN.
import subprocess

csv, xlsx = f"{OUT_DIR}/{BOOK}.csv", f"{OUT_DIR}/{BOOK}.xlsx"
for args in (["ocr-nom-align", "check", csv, "--xlsx", xlsx], ["ocr-nom-align", "score", csv, "--min", str(SCORE_MIN)]):
    r = subprocess.run(args, capture_output=True, text=True)
    print("$", " ".join(args), f"(exit {r.returncode})")
    print(r.stdout, r.stderr)

In [ ]:
# ENV — after a clean RUN: print the environment fingerprint and the package version for the report.
# Keep it with the dictionary files used (names and dates) and the command line printed by RUN.
import json

from ocr_flow import envinfo

import ocr_nom_align

PY = None
print("ocr-nom-align", ocr_nom_align.__version__)
print(json.dumps(envinfo.fingerprint(PY), indent=2))